In [1]:
!pip -q install ultralytics roboflow==1.*

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 19.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 75.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 46.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 100.3 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
api_key = userdata.get("ROBOFLOW_API_KEY")
workspace_name = userdata.get("ROBOFLOW_WORKSPACE")
project_name = userdata.get("ROBOFLOW_PROJECT")
version = userdata.get("ROBOFLOW_PROJECT_VERSION")

In [3]:
from roboflow import Roboflow
rf = Roboflow(api_key=api_key)
project = rf.workspace(workspace_name).project(project_name)
version = project.version(version)

dataset = version.download("yolov8")

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Construction-Site-Safety-2 in yolov8:: 100%|██████████| 1446/1446 [00:00<00:00, 2158.60it/s]


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [4]:
from ultralytics import YOLO

model = YOLO("yolo12n.pt")

In [5]:
from pathlib import Path

run_name = "yolo12_run_1"
project_dir = "yolo_runs"

model.train(
  data=f"{dataset.location}/data.yaml",
  epochs=80,              # enough to converge; early stop will cap if overfitting
  imgsz=768,
  batch=-1,               # drop to 8 if VRAM is tight; 'auto' also works
  patience=20,            # early stopping on val mAP
  optimizer="AdamW",      # tends to work well on small datasets
  lr0=0.003,              # slightly conservative start LR
  lrf=0.12,               # final LR fraction (cosine schedule)
  weight_decay=0.0005,
  warmup_epochs=3.0,
  cos_lr=True,
  seed=42,
  workers=2,              # stable on Colab/Kaggle
  amp=True,

  # --- Augmentations (balanced for tiny datasets) ---
  # mosaic=1.0,             # strong mix of contexts initially
  # mixup=0.10,             # mild label mixing
  # copy_paste=0.20,        # helps crowded PPE scenes
  # hsv_h=0.015, hsv_s=0.70, hsv_v=0.40,
  degrees=5.0,            # small rotations keep geometry realistic
  translate=0.05,         # 5% shifts
  scale=0.50,             # up/down-scale range
  # shear=2.0,
  # perspective=0.0005,
  fliplr=0.50,            # horizontal flips okay for PPE; keep flipud off
  flipud=0.0,

  # --- Overfit guards ---
  close_mosaic=10,        # turn mosaic OFF for the last 10 epochs
  freeze=10,              # freeze early backbone at start (stabilizes on tiny data)
  project=project_dir,           # <— set once
  name=run_name,                 # <— set once
  exist_ok=True,                 # don’t auto-increment exp numbers
)

run_dir = Path(project_dir) / run_name
best = run_dir / "weights/best.pt"
last = run_dir / "weights/last.pt"
print("BEST:", best.resolve())
print("LAST:", last.resolve())

Ultralytics 8.3.223 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/Construction-Site-Safety-2/data.yaml, degrees=5.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=80, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=10, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=768, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.003, lrf=0.12, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12n.pt, momentum=0.937, mosaic=1.0, multi_scale=False, name=yolo12_run_1, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patience=20, perspective=0.

In [6]:
!zip -r /content/yolo12_run_1.zip /content/yolo_runs/yolo12_run_1

from google.colab import files
files.download("/content/yolo12_run_1.zip")

  adding: content/yolo_runs/yolo12_run_1/ (stored 0%)
  adding: content/yolo_runs/yolo12_run_1/train_batch562.jpg (deflated 11%)
  adding: content/yolo_runs/yolo12_run_1/labels.jpg (deflated 22%)
  adding: content/yolo_runs/yolo12_run_1/BoxP_curve.png (deflated 10%)
  adding: content/yolo_runs/yolo12_run_1/results.png (deflated 7%)
  adding: content/yolo_runs/yolo12_run_1/weights/ (stored 0%)
  adding: content/yolo_runs/yolo12_run_1/weights/best.pt (deflated 11%)
  adding: content/yolo_runs/yolo12_run_1/weights/last.pt (deflated 11%)
  adding: content/yolo_runs/yolo12_run_1/confusion_matrix_normalized.png (deflated 12%)
  adding: content/yolo_runs/yolo12_run_1/args.yaml (deflated 52%)
  adding: content/yolo_runs/yolo12_run_1/train_batch1.jpg (deflated 3%)
  adding: content/yolo_runs/yolo12_run_1/BoxF1_curve.png (deflated 10%)
  adding: content/yolo_runs/yolo12_run_1/results.csv (deflated 60%)
  adding: content/yolo_runs/yolo12_run_1/val_batch0_pred.jpg (deflated 11%)
  adding: content/

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>